## Limit orders at a band

A Strategy that rests limit orders rather than taking the market: a buy at the lower edge
of a Bollinger band, moved to the band at every close, and once it fills a sale at the
band's middle. A limit fills only where a bar reaches it, at the maker's fee where it
rested and at the taker's where its first bar opened through it, and many never fill at
all. Hourly Bitcoin bars of the last quarter of 2025.

In [1]:
import pathlib

import exchange_router
import pandas
import probatum

print(f"probatum {probatum.__version__}, exchange-router {exchange_router.__version__}")

probatum 3.0.0, exchange-router 3.3.0


### The bars

Binance's hourly BTCUSDT, fetched once into `bars/`, cut to October to December 2025 by a
slice that keeps the bars before it.

In [2]:
PATH = pathlib.Path("../bars/binance-spot-btcusdt-1h-2024-2025")

if PATH.exists():
    hourly = probatum.Data.load(path=PATH)
else:
    with exchange_router.Router.local(exchanges=["binance"]) as router:
        hourly = probatum.Data.from_router(router, exchange="binance", market="spot",
                                           interval="1h", symbols=["BTCUSDT"],
                                           start="2024-01-01", end="2025-12-31")
    hourly.save(path=PATH)

bars = hourly["2025-10":"2025-12"]

bars

Data(BTCUSDT, 1h, 2208 bars from 2025-10-01 00:00 to 2025-12-31 23:00)

### The strategy

Every verb returns an `Order`, a live handle whose `status` says where it stands on any
later bar: `resting` while it waits on the book. `replace` moves a resting order under its
own id, and a buy still resting once the position is open is cancelled, so the book never
holds two of them.

In [3]:
class Band(probatum.Strategy):
    def __init__(self, length=48, deviations=2.0):
        self.length, self.deviations = length, deviations

    def init(self, data):
        bands       = probatum.ta.bbands(data.close, length=self.length,
                                         deviations=self.deviations)
        self.lower  = bands.lower
        self.middle = bands.middle
        self.warmup = self.length
        self.order  = None

    def next(self, bar, account):
        side  = "buy" if account.position == 0.0 else "sell"
        price = self.lower[bar.i] if side == "buy" else self.middle[bar.i]

        if self.order is not None and self.order.status == "resting":
            if self.order.side == side:
                account.replace(self.order, price=price)
                return
            account.cancel(self.order)

        if side == "buy":
            self.order = account.buy(size=account.equity * 0.9 / price, limit=price)
        else:
            self.order = account.sell(size=account.position, limit=price)

    def marks(self):
        return [probatum.chart.marks.Line(self.lower, name="lower band", width=1),
                probatum.chart.marks.Line(self.middle, name="middle", width=1,
                                          style="dashed")]


result = probatum.backtest(Band(), data=bars, venue="binance_spot")

result

,run,buy_and_hold,low,high,level,method,why_none
total_return_pct,-15.22,-23.36,,,,,1
cagr_pct,,,,,,,2
volatility_pct,28.67,46.02,,,,,1
sharpe,-2.14,-2.06,-5.99,1.70,0.95,"delta method, Newey-West lag 14, floored at lag zero",
max_drawdown_pct,20.87,34.76,,,,,1
exposure_pct,35.30,,,,,,
num_trades,30,,,,,,
win_rate,0.63,,,,,,3
profit_factor,0.50,,,,,,3
costs,486,,,,,,


### Maker and taker

Each fill says whether it took liquidity. A limit the next bar opened through filled at its
own price but paid the taker's fee; one the bar had to come to paid the maker's.

In [4]:
fills = result.fills.to_pandas()

fills.groupby(by=["side", "taker"]).agg(fills=("price", "size"), size=("size", "sum"),
                                        fees=("fee", "sum")).round(decimals=4)

,,fills,size,fees
side,taker,,,
buy,False,30,2.4816,243.6774
sell,False,30,2.4816,242.6414


### Orders that never filled

The order log keeps every event of every order, counted here by kind, and the last few
as the log holds them.

In [5]:
orders = result.orders.to_pandas()

print(orders["kind"].value_counts().to_string())

orders.tail(n=4)

kind
placed    2159
filled      60


,bar,time,kind,order,instrument,side,size,price,reason
2215,2203,2025-12-31 19:00:00+00:00,placed,61,BTCUSDT,buy,0.08816,86939.14,replace
2216,2204,2025-12-31 20:00:00+00:00,placed,61,BTCUSDT,buy,0.08816,86970.01,replace
2217,2205,2025-12-31 21:00:00+00:00,placed,61,BTCUSDT,buy,0.08816,86997.85,replace
2218,2206,2025-12-31 22:00:00+00:00,placed,61,BTCUSDT,buy,0.08816,87021.91,replace


### A fill and an order, told

`explain` gives one fill's price from its reference to its fee, and one order's life
event by event: here the filled order that was moved least often.

In [6]:
events = orders.groupby(by="order").size()
order  = events[fills["order"].unique()].idxmin()

print(result.explain(result.fills[int(fills.index[fills["order"] == order][0])]))
print()
print(result.explain(result.orders[int(orders.index[orders["order"] == order][0])]))

fill 15, BTCUSDT sell 0.07787 at 110,223.92, 2025-11-02 23:00
  reference        the limit's price              110,223.92
  spread           none for a maker                    +0.00
  slippage         none for a maker                    +0.00
  impact           none for a maker                    +0.00
  held in the bar  the bar's low was not reached       +0.00
  fee              maker 0.1% on 8,583.14               8.58

order 16, BTCUSDT sell limit 0.07787 at 110,223.92
  2025-11-02 22:00  placed  limit, sell 0.07787 at 110,223.92
  2025-11-02 23:00  filled  0.07787 at 110,223.92, fill 15


### The chart

In [7]:
result.chart(result.policy.marks(), focus=24 * 21,
             title="Limits at a 48-hour band, BTCUSDT 1h, binance spot, late 2025")

Chart(bars=2208, panels=3)